In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
train_dir = "/kaggle/input/eye-cataractmature-immature-normal/eye_cataracwithlabels/train"
valid_dir = "/kaggle/input/eye-cataractmature-immature-normal/eye_cataracwithlabels/train"
test_dir = "/kaggle/input/eye-cataractmature-immature-normal/eye_cataracwithlabels/test"

In [ ]:
train_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_resnet_v2.preprocess_input)
valid_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_resnet_v2.preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.inception_resnet_v2.preprocess_input)

train_batches = train_datagen.flow_from_directory(train_dir, target_size=(299, 299), batch_size=50, class_mode='categorical')
valid_batches = valid_datagen.flow_from_directory(valid_dir, target_size=(299, 299), batch_size=50, class_mode='categorical')
test_batches = test_datagen.flow_from_directory(test_dir, target_size=(299, 299), batch_size=50, class_mode='categorical', shuffle=False)

In [ ]:
imgs, labels = next(train_batches)

In [ ]:
base_model = InceptionResNetV2(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.summary()
print(type(base_model))

In [ ]:
model = models.Model(inputs=base_model.input, outputs=base_model.output)
model.trainable = False

# Add GlobalAveragePooling2D and Dense layer for classification
x = layers.GlobalAveragePooling2D()(model.output)
x = layers.Dense(3, activation='softmax')(x)

# Final model
final_model = models.Model(inputs=model.input, outputs=x)

In [ ]:
final_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

In [ ]:
history = final_model.fit(train_batches, validation_data=valid_batches, steps_per_epoch=70, validation_steps=70, epochs=30, verbose=1, callbacks=[early_stopping])

In [ ]:
# Plot accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
test_loss, test_accuracy = final_model.evaluate(test_batches, verbose=1)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Get the class labels
class_labels = list(test_batches.class_indices.keys())

# Reset the test_batches generator
test_batches.reset()

# Predict on the test data
predictions = final_model.predict(test_batches, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

# True labels
true_classes = test_batches.classes

# Confusion matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)
print('Confusion Matrix')
print(conf_matrix)

# Classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print('Classification Report')
print(report)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
def load_and_preprocess_image(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.inception_resnet_v2.preprocess_input(img_array)
    return img_array

image_path = "/kaggle/input/eye-cataractmature-immature-normal/eye_cataracwithlabels/test/Immature/Immature (1).jpg"
img_array = load_and_preprocess_image(image_path)
single_prediction = final_model.predict(img_array)
predicted_class = np.argmax(single_prediction, axis=1)
predicted_label = class_labels[predicted_class[0]]

print(f"Predicted Label for {image_path}: {predicted_label}")